# Step 4: Casting parameters

The casting parameters describe the caster's motion: how the rod butt
rotates, and how the hand moves, through the stroke. In the library the
rotation is a function `theta(t)` that gives the rod-butt angle [rad],
measured from the horizontal `+x` axis, at time `t` [s].

**Limits of this step today.**

- A stroke cannot be loaded from a file, and there is no stroke file
  format yet.
- The Cast #1 simulation has its stroke built in. You cannot pass your
  own stroke to `simulate_cast1`.
- The hand translation (the haul path) of Cast #1 is not part of the
  public API, so this notebook shows the rotation only.
- The generic cast `simulate_cast` takes two stroke parameters: the sweep
  angle and the stroke duration.

The `flycastsim` workflow has five steps. Step 1 stands on its own, and
steps 2 to 4 each produce parameters that step 5 uses.

1. Simple 1-D model (`01_simple_1d_model.ipynb`)
2. Rod characteristics (`02_rod_characteristics.ipynb`)
3. Line and leader characteristics (`03_line_and_leader.ipynb`)
4. Casting parameters (`04_casting_parameters.ipynb`)
5. Simulate a fly cast (`05_simulate_cast.ipynb`)

`complete_workflow.ipynb` runs steps 2 to 5 in sequence.

In [ ]:
import numpy as np
import plotly.graph_objects as go

from flycastsim.fem import cast1_stroke, casting_stroke

## The idealised stroke

`casting_stroke(sweep, t_stroke)` sweeps the rod butt smoothly through
the angle `sweep` in the time `t_stroke` and then holds it. The angular
velocity is zero at both ends.

In [ ]:
t = np.linspace(0.0, 0.8, 401)
fig = go.Figure()
for sweep_deg, t_stroke in ((120, 0.4), (120, 0.25), (90, 0.4)):
    theta = casting_stroke(np.radians(sweep_deg), t_stroke)
    angle = np.degrees([theta(ti) for ti in t])
    fig.add_trace(go.Scatter(x=t, y=angle,
                             name=f"sweep {sweep_deg}°, {t_stroke} s"))
fig.update_layout(title="Idealised casting stroke",
                  xaxis_title="time [s]",
                  yaxis_title="rod-butt angle from horizontal [deg]")
fig

## The Cast #1 stroke

`cast1_stroke()` is the rod-butt sweep fitted by eye to the footage of
Cast #1 from *The Rod & The Cast* (Løvoll & Borger). Time is relative to
RSP (Rod Straight Position), so `t = 0` is RSP. The rod starts up and
back and sweeps forward.

In [ ]:
t1 = np.linspace(-0.4, 1.0, 701)
theta1 = cast1_stroke()
angle1 = np.array([theta1(ti) for ti in t1])

fig = go.Figure(go.Scatter(x=t1, y=np.degrees(angle1), name="Cast #1"))
fig.add_vline(x=0.0, line_dash="dash", annotation_text="RSP")
fig.update_layout(title="Cast #1 stroke",
                  xaxis_title="time relative to RSP [s]",
                  yaxis_title="rod-butt angle from horizontal [deg]")
fig

The angular velocity shows where the caster accelerates and stops the
rod.

In [ ]:
omega1 = np.gradient(angle1, t1)
fig = go.Figure(go.Scatter(x=t1, y=np.degrees(omega1)))
fig.add_vline(x=0.0, line_dash="dash", annotation_text="RSP")
fig.update_layout(title="Cast #1 angular velocity",
                  xaxis_title="time relative to RSP [s]",
                  yaxis_title="angular velocity [deg/s]")
fig

In [ ]:
peak = np.argmax(np.abs(omega1))
print(f"Total sweep: {np.degrees(angle1.max() - angle1.min()):.0f}°")
print(f"Peak angular velocity: {abs(np.degrees(omega1[peak])):.0f}°/s "
      f"at t = {t1[peak]:+.3f} s")